# Tutorial 3: Private Neural Networks & Transformer / LLM Inference (`prix.nn`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mugurbil/prix/blob/main/examples/03_private_transformers_and_llms.ipynb)

**PriX** includes a PyTorch/CrypTen-style neural network module library (`prix.nn`) designed specifically for private Transformer and LLM research:
- **Linear & Convolutions**: `nn.Linear`, `nn.Conv1d`, `nn.Conv2d`, `nn.Embedding`
- **Normalizations**: `nn.LayerNorm`, `nn.RMSNorm`, `nn.BatchNorm1d`, `nn.BatchNorm2d`
- **Activations**: `nn.GELU`, `nn.SiLU`, `nn.ReLU`, `nn.Softmax`, `nn.LogSoftmax`, `nn.Sigmoid`, `nn.Tanh`
- **Attention & Containers**: `nn.MultiheadAttention`, `nn.Sequential`, `nn.ModuleList`, `nn.ModuleDict`

In this notebook, you will:
1. Build a custom pre-norm **Transformer Encoder Block** using `prix.nn.Module`.
2. Encrypt model weights on **Party 0** (`.encrypt(src=0)`) and run private inference on secret-shared token embeddings from **Party 1** (`src=1`).
3. Compare private Transformer inference across `"lut"`, `"shaft"`, `"crypten"`, and `"fission"` modes.

In [ ]:
!pip install -q prix-jax

## 1. Building & Encrypting a Custom Transformer Block with `prix.nn.Module`

Just like `torch.nn.Module` / `crypten.nn.Module`, any `prix.nn.Module` can be evaluated in plaintext first and then secret-shared across all MPC parties with a single `.encrypt(src=0)` call:

In [ ]:
import time
import numpy as np
import prix
import prix.nn as nn

class PrivateTransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, d_ff=256):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        # Pre-LayerNorm Self-Attention + FFN residual block
        h = self.ln1(x)
        x = x + self.attn(h, h, h)
        x = x + self.ffn(self.ln2(x))
        return x

def benchmark_transformer_block():
    # Party 0 holds the model weights; Party 1 holds the input sequence (batch=1, seq=16, d_model=64)
    block = PrivateTransformerBlock(d_model=64, n_heads=4, d_ff=256).encrypt(src=0)
    x = prix.randn(1, 16, 64)

    # Warmup XLA compilation
    _ = block(x).get_plain_text()

    t0 = time.perf_counter()
    out = block(x).get_plain_text()
    elapsed_ms = (time.perf_counter() - t0) * 1e3
    return out.shape, elapsed_ms

for mode in ["crypten", "shaft", "lut", "fission"]:
    shape, ms = prix.run(benchmark_transformer_block, world_size=2, mode=mode)
    print(f"[{mode:>7}] PrivateTransformerBlock output={shape} | latency={ms:.2f} ms")

## 2. Loading Custom Weights (`state_dict` / `load_state_dict`)

Researchers often train or fine-tune models in PyTorch/JAX/Flax and then load the weights into PriX for private evaluation. Every `prix.nn.Module` supports `.state_dict()` and `.load_state_dict(...)` prior to calling `.encrypt()`:

In [ ]:
@prix.mpc.run_multiprocess(world_size=2, mode="shaft")
def state_dict_demo():
    model = nn.Sequential(
        nn.Linear(8, 16),
        nn.SiLU(),
        nn.RMSNorm(16),
        nn.Linear(16, 4),
    )
    # Inspect parameter names & shapes before encrypting
    sd = model.state_dict()
    prix.print("Parameter keys:", list(sd.keys()))

    # Encrypt model on Party 0 and evaluate on Party 1's input
    model.encrypt(src=0)
    x = prix.cryptensor(np.ones((2, 8), dtype=np.float32), src=1)
    return model(x).get_plain_text()

print("Encrypted Sequential output:\n", state_dict_demo())